> Historical one-unit HTTP interface and CH1 evidence. Use `modular_dac_acceptance.ipynb` for the current multi-unit dashboard and configured meter channel.

# Local dashboard acceptance — live DAC + Keysight

These cells exercise the **same HTTP actions and state shown by the webpage**, with the assertions written here. Keep Chrome open at http://localhost:8765 while running. Browser layout/visibility still requires a visual check; HTTP evidence alone does not verify pixels.

CH1 is connected to the Keysight 34461A. CH2–4 have register/telemetry evidence only. All outputs must be isolated. Tests set CH1 between −9 and +10 V, use small ramps, and exercise USB recovery on all four channels. The server owns both instruments; do not open a second driver connection. Physical button contacts are not emulated by these tests; the USB commands invoke their firmware actions.

Starting the dashboard is read-only. This notebook explicitly initializes cold outputs before testing. A DAC register may contain a nonzero code while the analog output is still clamped after a restart. **Only the Keysight is an independent analog reading.**

In [ ]:
RUN_LIVE = False  # Set True only on the isolated bench.
ALL_OUTPUTS_ISOLATED = False
BASE = "http://127.0.0.1:8765"
TOLERANCE_V = 0.0005


In [ ]:
import json, time, uuid, re, math, statistics
from pathlib import Path
from datetime import datetime, timezone
from urllib.request import Request, urlopen
from urllib.error import HTTPError
import matplotlib.pyplot as plt

assert RUN_LIVE and ALL_OUTPUTS_ISOLATED, "Enable the two explicit live-test flags first"
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'malab/dashboard').is_dir())
results = root/'malab/notebooks/results'/('dashboard_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
results.mkdir(parents=True)
html = urlopen(BASE, timeout=3).read().decode()
token = re.search(r'name="dac-token" content="([^"]+)"', html).group(1)
def state():
    return json.load(urlopen(BASE+'/api/state', timeout=3))
def wait_for(predicate, timeout=20):
    deadline=time.monotonic()+timeout
    while time.monotonic()<deadline:
        s=state()
        if predicate(s):return s
        time.sleep(.05)
    raise AssertionError('State deadline expired: '+json.dumps({k:s[k] for k in ('network','recovery','ramp','pending')}))
def action(name, **fields):
    payload=dict(action=name, request_id=str(uuid.uuid4()), **fields)
    req=Request(BASE+'/api/action', json.dumps(payload).encode(),
                headers={'Content-Type':'application/json','X-DAC-Token':token})
    job=json.load(urlopen(req,timeout=7))
    s=wait_for(lambda s:s['jobs'].get(job['id'],{}).get('state') in ('complete','failed'))
    job=s['jobs'][job['id']]
    assert job['state']=='complete', job
    return job
def fresh(s):
    assert s['network']['state']=='good', s['network']
    assert s['now']-s['network']['last_ok']<2.5
    assert s['meter']['state']=='good', s['meter']
    assert s['now']-s['meter']['at']<2.5
    assert not s['device_error'],s['device_error']
def codes(s):return [c['code'] for c in s['channels']]
def readings(count=5):
    rows=[];previous=None
    while len(rows)<count:
        s=state();fresh(s)
        if s['meter']['at']!=previous:
            rows.append(s);previous=s['meter']['at']
        time.sleep(.04)
    return rows
def save(name,value):
    (results/(name+'.json')).write_text(json.dumps(value,indent=2),encoding='utf-8')
print('Evidence directory:',results)


## D01 — Four channels, fresh response indicator, SET/GET versus meter
Initialize explicitly. Check all four cards' data, unchanged CH2–4, and both requested voltage and displayed register voltage against the independent CH1 measurement (±0.5 mV).

In [ ]:
s=wait_for(lambda s:s['network']['state']=='good' and s['meter']['state']=='good')
action('initialize')
time.sleep(.4)
baseline=state(); fresh(baseline)
assert [c['channel'] for c in baseline['channels']]==[1,2,3,4]
assert [c['calibrated'] for c in baseline['channels']]==[True,False,False,False]
others=codes(baseline)[1:]
set_results=[]
for target in [-9,-5,-1,-.05,0,.05,1,5,9,10,0]:
    action('set',channel=1,voltage=target)
    time.sleep(.4)
    samples=readings()
    meter=statistics.mean(s['meter']['voltage'] for s in samples)
    displayed=samples[-1]['channels'][0]['voltage']
    row=dict(target=target,displayed=displayed,meter=meter,
             request_error_mV=(meter-target)*1000,display_error_mV=(meter-displayed)*1000)
    set_results.append(row);save('D01',set_results)
    print(row)
    assert abs(meter-target)<=TOLERANCE_V,row
    assert abs(meter-displayed)<=TOLERANCE_V,row
    assert codes(samples[-1])[1:]==others
print('PASS: 11 points, four channel records, fresh network/meter, ±0.5 mV')


## D02 — Ramp timing, path and endpoints while the dashboard polls
Three speeds differing by 10×. Each ramp spans about five seconds, then runs in reverse. The 10-code staircase and asynchronous meter sampling are accounted for explicitly. Timing allowance is 5% plus 0.6 s for command/status sampling; this is not a hard real-time guarantee.

In [ ]:
ramp_results=[];ramp_traces=[]
for speed,span in [(.001,.005),(.01,.05),(.1,.5)]:
    for start,target in [(0,span),(span,0)]:
        action('set',channel=1,voltage=start);time.sleep(.35)
        initial=state();fresh(initial)
        start_v=initial['channels'][0]['voltage'];step_v=.000076283553*10
        target_code=round((target+9.992845256)/.000076283553)
        delay_us=math.ceil(step_v/speed*1e6)
        expected=math.ceil(abs(target_code-initial['channels'][0]['code'])/10)*delay_us/1e6
        started=time.time()
        action('ramp',channel=1,voltage=target,speed=speed,step_size=10)
        trace=[];deadline=time.monotonic()+expected+8
        while time.monotonic()<deadline:
            s=state();fresh(s);trace.append({k:s[k] for k in ('now','channels','meter','network','recovery','ramp','device_error')})
            assert codes(s)[1:]==others
            if s['ramp']['state']=='COMPLETE':break
            assert s['ramp']['state']=='RUNNING',s['ramp']
            time.sleep(.05)
        assert trace[-1]['ramp']['state']=='COMPLETE'
        duration=time.time()-started
        active=[s for s in trace if s['ramp']['state']=='RUNNING']
        assert len({s['channels'][0]['code'] for s in active})>=3,'Telemetry must change during the ramp'
        assert any(s['channels'][0]['source']=='write' for s in active)  # Starting code can still be checked readback.
        direction=1 if target>start else -1
        previous=start_v;meter_previous=None
        for s in active:
            v=s['channels'][0]['voltage'];m=s['meter']['voltage']
            assert direction*(v-previous)>=-1e-9,'Device path reversed'
            assert min(start_v,target)-step_v<=v<=max(start_v,target)+step_v
            # Independent samples can be ~0.4 s apart across the polling loops.
            assert abs(m-v)<=step_v+speed*.6+TOLERANCE_V,(m,v)
            if meter_previous is not None:assert direction*(m-meter_previous)>=-TOLERANCE_V
            previous=v;meter_previous=m
        assert abs(duration-expected)<=expected*.05+.6,(duration,expected)
        time.sleep(.3);end=readings(3)
        measured=statistics.mean(s['meter']['voltage'] for s in end)
        assert abs(measured-target)<=TOLERANCE_V
        row=dict(speed=speed,start=start,target=target,expected_s=expected,observed_s=duration,meter_endpoint=measured)
        ramp_results.append(row);ramp_traces.append(dict(summary=row,samples=trace))
        save('D02',ramp_traces);print(row)
fig,axes=plt.subplots(3,1,figsize=(10,9))
for ax,entry in zip(axes,ramp_traces[::2]):
    trace=entry['samples'];t0=trace[0]['now']
    ax.plot([s['channels'][0]['at']-t0 for s in trace],[s['channels'][0]['voltage'] for s in trace],label='DAC 1 device value')
    ax.plot([s['meter']['at']-t0 for s in trace],[s['meter']['voltage'] for s in trace],label='Keysight',alpha=.7)
    ax.set(xlabel='Seconds',ylabel='V',title=str(entry['summary']['speed'])+' V/s');ax.legend()
fig.tight_layout();fig.savefig(results/'ramps.png',dpi=160);plt.show()
print('PASS: six ramps, live telemetry, monotonic paths, timing and endpoints')


## D03 — Network reset preserves every channel; manual disconnect holds
The green signal requires a fresh application reply, not merely a connected socket. A manual disconnect stays disconnected; reconnect resumes monitoring. Stop-ramp must hold the intermediate voltage.

In [ ]:
for ch,v in [(1,.05),(2,.01),(3,-.01),(4,.02)]:action('set',channel=ch,voltage=v)
time.sleep(.4);before=state();fresh(before)
reset=action('network')
after=wait_for(lambda s:s['network']['state']=='good' and s['network']['last_ok']>reset['finished'])
assert codes(after)==codes(before),(codes(before),codes(after))
assert abs(after['meter']['voltage']-.05)<=TOLERANCE_V
action('disconnect');time.sleep(.8);disconnected=state()
assert disconnected['network']['state']=='disconnected'
last_ok=disconnected['network']['last_ok']
time.sleep(2.6);held=state()
assert held['network']['last_ok']==last_ok and held['network']['state']=='disconnected'
assert held['now']-last_ok>2.5
assert abs(held['meter']['voltage']-.05)<=TOLERANCE_V
action('connect');after=wait_for(lambda s:s['network']['state']=='good')
assert codes(after)==codes(before)
action('ramp',channel=1,voltage=0,speed=.01)
time.sleep(1.3);mid=state();assert mid['ramp']['state']=='RUNNING'
action('abort');held=wait_for(lambda s:s['network']['state']=='good' and s['ramp']['state']=='ABORTED')
time.sleep(.5);held=state();held_code=held['channels'][0]['code'];held_voltage=held['channels'][0]['voltage']
assert 0<held_voltage<.05
time.sleep(.8);later=state();assert later['channels'][0]['code']==held_code
assert abs(later['meter']['voltage']-held_voltage)<=TOLERANCE_V
save('D03',dict(before=before,after_reset=after,disconnected=disconnected,held=later))
print('PASS: reset holds all four codes, disconnected values age, stop holds intermediate output')


## D04 — SAFE ZERO through independent USB, both signs, all channels
Disconnect TCP before emulating the button. The dashboard must show progress from USB telemetry, reach all four zero codes, remain latched, then rearm only after NETWORK. CH1 analog slope and endpoint are measured; other channels are checked by device codes.

In [ ]:
safe_results=[]
for sign in [1,-1]:
    for ch,v in [(1,sign*.05),(2,.01),(3,-.01),(4,.02)]:action('set',channel=ch,voltage=v)
    time.sleep(.35);before=state()
    action('disconnect')
    started=time.time();action('safe_zero')
    trace=[];deadline=time.monotonic()+10
    while time.monotonic()<deadline:
        s=state();trace.append({k:s[k] for k in ('now','channels','meter','network','recovery','ramp','device_error')})
        assert s['network']['state']!='good'
        assert s['recovery']['state'] in ('ACTIVE','LATCHED'),s['recovery']
        if s['recovery']['state']=='LATCHED':break
        time.sleep(.05)
    assert trace[-1]['recovery']['state']=='LATCHED'
    duration=time.time()-started
    assert 4.9<=duration<=6.5,duration
    assert codes(trace[-1])==[130996,131072,131072,131072]
    active=[s for s in trace if s['recovery']['state']=='ACTIVE']
    assert len({s['channels'][0]['code'] for s in active})>=4
    unique={s['meter']['at']:s['meter']['voltage'] for s in active}
    points=sorted(unique.items())
    # Fit only the middle of the measured ramp, avoiding start/endpoint sampling.
    points=[(t,v) for t,v in points if .005<abs(v)<.045]
    assert len(points)>8
    tmean=statistics.mean(t for t,v in points);vmean=statistics.mean(v for t,v in points)
    slope=sum((t-tmean)*(v-vmean) for t,v in points)/sum((t-tmean)**2 for t,v in points)
    assert .0095<=abs(slope)<=.0101 and sign*slope<0,slope
    time.sleep(.4);latched=state()
    assert latched['recovery']['state']=='LATCHED' and codes(latched)==[130996,131072,131072,131072]
    assert abs(latched['meter']['voltage'])<=TOLERANCE_V
    safe_results.append(dict(sign=sign,duration_s=duration,slope_V_per_s=slope,final=latched,samples=trace))
    save('D04',safe_results)
    action('network');rearmed=wait_for(lambda s:s['network']['state']=='good' and s['recovery']['state']=='READY')
    assert codes(rearmed)==[130996,131072,131072,131072]
    print(dict(sign=sign,duration_s=duration,slope_V_per_s=slope,meter_zero=latched['meter']['voltage']))
final=state();fresh(final);save('final_state',final)
print('PASS: SAFE ZERO independent of TCP, measured CH1 slope/zero, four zero targets, latch and rearm')
print('Final meter:',final['meter']['voltage'],'V; network:',final['network']['state'])


## Chrome visual check (manual)
Confirm all four DAC cards, live Keysight value, history traces, and the green **Network good** signal are visible. During D02 the CH1 device and meter values change, output buttons are disabled and Stop ramp is available. During D03 the signal leaves green when disconnected and returns after a fresh response. During D04 it shows network paused and safe-zero progress, then the latched state. Reload the page to receive UI updates. The automatic evidence above validates the data and actions; record this visual check separately.